# FireSight-IR | Module 1a — VIIRS & FIRMS Cloud Ingestion

**Project:** FireSight-IR — Physics-constrained wildfire intelligence at constellation scale  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Platform:** Google Colab (cloud-native, no local downloads)  
**Guide version:** aligned with Module 1a Team Guide v2.0, March 2026  

---

## What this notebook does

1. **Mounts Google Drive** for persistent output storage across Colab sessions
2. **Downloads FIRMS** VIIRS S-NPP Standard Processing fire detection archives (2018–2023) via the FIRMS area API — saved as parquet, one file per year
3. **Streams VNP14IMG** Collection 2 granules from NASA Earthdata S3 — guided by FIRMS active-fire days to reduce data volume ~8×
4. **Extracts fire pixel records** (BT_I4, BT_I5, BTD, FRP, geometry, background stats) into parquet files — one row per fire pixel
5. **Verifies output quality** against expected statistics from the team guide

> **Note:** 32×32 spatial patch extraction happens in Module 1c. This notebook produces the per-pixel tabular dataset that feeds Modules 1b (ERA5 co-location) and 2 (feature engineering).

---

## Before running

You need two free accounts:
- **NASA Earthdata:** https://urs.earthdata.nasa.gov/users/new
- **FIRMS map key:** https://firms.modaps.eosdis.nasa.gov/api/map_key/ (log in with Earthdata account)

---

## Section 0 — Install packages and mount Drive

**Run this section at the start of every Colab session.** Packages are wiped on disconnect.

In [ ]:
# ── Install packages (run every session) ─────────────────────────────────────
!pip install earthaccess requests pandas numpy xarray h5py h5netcdf pyarrow tqdm -q
print("✓ Packages installed")

In [ ]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
# Files saved to Drive persist across sessions. Colab environment does not.
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os
import time
import warnings
import requests
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import xarray as xr
import h5py
import earthaccess
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')  # suppress xarray phony_dims warnings
print("✓ Imports complete")

---
## Section 1 — Project configuration

All parameters live here. **Do not hardcode values anywhere else in the notebook.**

In [ ]:
# ── Spatial domain — Western US (CA + NV + OR + parts of WA, ID, AZ, UT) ─────
# Three formats required because different libraries use different conventions

BBOX_TUPLE = (-125, 32, -109, 49)               # (west, south, east, north) — earthaccess
BBOX_STR   = '-125,32,-109,49'                  # 'west,south,east,north'    — FIRMS API URL
BBOX_DICT  = {                                   # dict of floats             — DataFrame filtering
    'lon_min': -125, 'lon_max': -109,
    'lat_min':   32, 'lat_max':   49,
}
# CRITICAL: If you change one format, change all three.

# ── Temporal domain ───────────────────────────────────────────────────────────
TRAIN_YEARS        = [2018, 2019, 2020, 2021, 2022]  # training data
VAL_YEAR           = 2023                             # held-out validation — never touch during training
ALL_YEARS          = TRAIN_YEARS + [VAL_YEAR]
FIRE_SEASON_MONTHS = [5, 6, 7, 8, 9, 10]             # May–Oct: >95% of western US fire activity

# ── Data sources ──────────────────────────────────────────────────────────────
FIRMS_SOURCE       = 'VIIRS_SNPP_SP'   # Standard Processing — quality-controlled archive
                                        # Do NOT use 'VIIRS_SNPP_NRT' (preliminary product)
VIIRS_SHORT_NAME   = 'VNP14IMG'        # VIIRS/NPP I-Band 375m Active Fire, Collection 2
VIIRS_VERSION      = '002'             # Collection 2 — always use this, not 001

# ── FIRMS-guided VIIRS download filter ───────────────────────────────────────
# Only process VIIRS granules on days with at least this many FIRMS detections
# in the study region. Eliminates ~85% of granules with no significant fire activity.
MIN_FIRMS_COUNT = 10

# ── Your FIRMS map key ────────────────────────────────────────────────────────
# Get from: https://firms.modaps.eosdis.nasa.gov/api/map_key/
# Limit: 5,000 requests per 10 minutes
FIRMS_MAP_KEY = 'YOUR_FIRMS_MAP_KEY_HERE'  # <-- replace this

# ── Output paths on Google Drive ─────────────────────────────────────────────
DRIVE_ROOT  = Path('/content/drive/MyDrive/firesight-ir')
FIRMS_DIR   = DRIVE_ROOT / 'data/raw/firms'
VIIRS_DIR   = DRIVE_ROOT / 'data/processed/viirs_fp'
FIGURES_DIR = DRIVE_ROOT / 'figures'

for d in [FIRMS_DIR, VIIRS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("✓ Configuration set")
print(f"  Training years : {TRAIN_YEARS}")
print(f"  Validation year: {VAL_YEAR}  (held out — do not use for training)")
print(f"  Bounding box   : {BBOX_STR}")
print(f"  FIRMS source   : {FIRMS_SOURCE}")
print(f"  VIIRS product  : {VIIRS_SHORT_NAME} v{VIIRS_VERSION}")
print(f"  Drive root     : {DRIVE_ROOT}")

---
## Section 2 — NASA Earthdata authentication

**Run every session.** Credentials are not persisted in Colab.

In [ ]:
# ── Authenticate to NASA Earthdata ────────────────────────────────────────────
# Prompts for username and password each session.
# Your credentials are never stored in the notebook or on Drive.
auth = earthaccess.login(strategy='interactive')

assert auth.authenticated, (
    "Authentication failed. "
    "Verify credentials at https://urs.earthdata.nasa.gov"
)
print(f"✓ Authenticated as: {auth.username}")
print("✓ S3 direct access enabled — no files will be written to Colab disk")

---
## Section 3 — FIRMS download

### How the FIRMS API works

The FIRMS area API returns fire detections as CSV for a bounding box, time period, and source.
The URL structure is:
```
https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{SOURCE}/{BBOX}/{DAY_RANGE}/{DATE}
```
**`DAY_RANGE` is capped at 5.** The `fetch_firms_year()` function below loops in 5-day
increments automatically. Never request more than 5 days per call.

In [ ]:
# ── FIRMS fetch function ───────────────────────────────────────────────────────
FIRMS_BASE = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv'

def fetch_firms_year(
    year: int,
    map_key: str,
    source: str,
    bbox_str: str,
    fire_season_months: list,
    day_chunk: int = 5,
    retry_delay: float = 2.0,
    max_retries: int = 3,
) -> pd.DataFrame:
    """
    Download a full year of FIRMS fire detections for a bounding box.
    
    Loops through the year in `day_chunk`-day windows (max 5, API limit).
    Filters to fire_season_months after concatenation.
    Returns a single DataFrame with all detections.

    Parameters
    ----------
    year               : calendar year (int)
    map_key            : FIRMS API map key (str)
    source             : FIRMS source string, e.g. 'VIIRS_SNPP_SP'
    bbox_str           : 'west,south,east,north' string
    fire_season_months : list of month ints to retain (e.g. [5,6,7,8,9,10])
    day_chunk          : days per API call (max 5)
    retry_delay        : seconds to wait before retrying on error
    max_retries        : maximum retries per chunk

    Returns
    -------
    pd.DataFrame with FIRMS columns plus 'month' column
    """
    assert day_chunk <= 5, "FIRMS API day_range cap is 5 — do not exceed"

    start_date = pd.Timestamp(f'{year}-01-01')
    end_date   = pd.Timestamp(f'{year}-12-31')
    chunks = pd.date_range(start=start_date, end=end_date, freq=f'{day_chunk}D')

    frames = []
    failed_chunks = []

    for chunk_start in tqdm(chunks, desc=f'FIRMS {year}', unit='chunk', leave=False):
        date_str = chunk_start.strftime('%Y-%m-%d')
        url = f"{FIRMS_BASE}/{map_key}/{source}/{bbox_str}/{day_chunk}/{date_str}"

        for attempt in range(max_retries):
            try:
                resp = requests.get(url, timeout=30)
                if resp.status_code == 200 and len(resp.text.strip()) > 0:
                    df = pd.read_csv(StringIO(resp.text))
                    if len(df) > 0 and 'latitude' in df.columns:
                        frames.append(df)
                    break
                elif resp.status_code == 400:
                    # 400 often means empty result for this date range — not an error
                    break
                else:
                    if attempt < max_retries - 1:
                        time.sleep(retry_delay)
            except requests.RequestException as e:
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                else:
                    failed_chunks.append(date_str)

    if not frames:
        print(f"  [WARN] {year}: no data returned — check map key and source")
        return pd.DataFrame()

    result = pd.concat(frames, ignore_index=True)

    # Parse date and filter to fire season
    result['acq_date'] = pd.to_datetime(result['acq_date'])
    result['month']    = result['acq_date'].dt.month
    result = result[result['month'].isin(fire_season_months)].copy()

    # Apply bounding box filter (FIRMS returns slight overestimates at bbox edges)
    result = result[
        (result['longitude'] >= BBOX_DICT['lon_min']) &
        (result['longitude'] <= BBOX_DICT['lon_max']) &
        (result['latitude']  >= BBOX_DICT['lat_min']) &
        (result['latitude']  <= BBOX_DICT['lat_max'])
    ].copy()

    if failed_chunks:
        print(f"  [WARN] {year}: {len(failed_chunks)} chunks failed: {failed_chunks[:5]}")

    return result


print("✓ fetch_firms_year() defined")

In [ ]:
# ── Download FIRMS for all years ──────────────────────────────────────────────
# Skips years already saved to Drive — safe to rerun after Colab disconnect.

assert FIRMS_MAP_KEY != 'YOUR_FIRMS_MAP_KEY_HERE', (
    "Set your FIRMS map key in the config cell (Section 1)"
)

firms_summary = {}

for year in ALL_YEARS:
    out_path = FIRMS_DIR / f'firms_viirs_snpp_{year}.parquet'

    if out_path.exists():
        df = pd.read_parquet(out_path)
        print(f"  {year}: {len(df):>9,} detections  [already on Drive — skipped]")
        firms_summary[year] = len(df)
        continue

    print(f"── {year} ──")
    df = fetch_firms_year(
        year               = year,
        map_key            = FIRMS_MAP_KEY,
        source             = FIRMS_SOURCE,
        bbox_str           = BBOX_STR,
        fire_season_months = FIRE_SEASON_MONTHS,
    )

    if len(df) > 0:
        df.to_parquet(out_path, index=False)
        print(f"  {year}: {len(df):>9,} detections saved → Drive")
        firms_summary[year] = len(df)
    else:
        print(f"  {year}: no data — check map key")
        firms_summary[year] = 0

print("\n✓ FIRMS download complete")

In [ ]:
# ── Inspect FIRMS output ──────────────────────────────────────────────────────
# Load 2020 (peak fire year) and inspect columns, value ranges, type distribution

firms_2020_path = FIRMS_DIR / 'firms_viirs_snpp_2020.parquet'
if firms_2020_path.exists():
    df20 = pd.read_parquet(firms_2020_path)

    print(f"2020 FIRMS: {len(df20):,} rows × {len(df20.columns)} columns")
    print(f"\nColumns: {list(df20.columns)}")
    print(f"\nDate range: {df20['acq_date'].min()} → {df20['acq_date'].max()}")

    # Type distribution — critical for label construction
    print("\ntype column distribution:")
    print(df20['type'].value_counts().to_string())
    print("  → type=0: vegetation fire (positive wildfire label)")
    print("  → type=2: static land source — gas flares, industrial (false-alarm training)")
    print("  → type=3: offshore (exclude from training)")

    print(f"\nBT_I4 (bright_ti4) range: {df20['bright_ti4'].min():.1f} – {df20['bright_ti4'].max():.1f} K")
    print(f"FRP range: {df20['frp'].min():.2f} – {df20['frp'].max():.1f} MW")
    print(f"\nconfidence distribution:")
    print(df20['confidence'].value_counts().to_string())
else:
    print("2020 FIRMS file not found — check download completed above")

---
## Section 4 — FIRMS-guided active fire day selection

We use FIRMS to identify days with meaningful fire activity before touching VIIRS granules.
Days with fewer than `MIN_FIRMS_COUNT` detections in the study region are skipped entirely.
This reduces VIIRS granule processing by ~8× compared to pulling all days.

In [ ]:
def get_active_fire_days(year: int, firms_dir: Path, min_count: int = 10) -> list:
    """
    Return a sorted list of date strings ('YYYY-MM-DD') for days with
    at least `min_count` FIRMS fire detections in the study region.

    Parameters
    ----------
    year      : calendar year
    firms_dir : Path to directory containing firms_viirs_snpp_{year}.parquet
    min_count : minimum FIRMS detections to qualify as an active fire day

    Returns
    -------
    list of 'YYYY-MM-DD' strings, sorted ascending
    """
    path = firms_dir / f'firms_viirs_snpp_{year}.parquet'
    if not path.exists():
        print(f"  [WARN] FIRMS file missing for {year}: {path}")
        return []

    df = pd.read_parquet(path, columns=['acq_date', 'latitude'])
    df['acq_date'] = pd.to_datetime(df['acq_date']).dt.strftime('%Y-%m-%d')

    # Count detections per day and keep only active fire days
    day_counts = df.groupby('acq_date').size()
    active_days = sorted(day_counts[day_counts >= min_count].index.tolist())

    return active_days


# Preview active fire day counts per year
print(f"Active fire day selection (MIN_FIRMS_COUNT = {MIN_FIRMS_COUNT})")
print(f"{'Year':<8} {'Active days':<14} {'Skipped days':<14}")
print('─' * 40)

for year in ALL_YEARS:
    active = get_active_fire_days(year, FIRMS_DIR, MIN_FIRMS_COUNT)
    season_days = len(FIRE_SEASON_MONTHS) * 30  # approximate
    skipped = season_days - len(active)
    flag = '← validation (do not use for training)' if year == VAL_YEAR else ''
    print(f"{year:<8} {len(active):<14} {skipped:<14} {flag}")

print("✓ FIRMS-guided day selection defined")

---
## Section 5 — VNP14IMG structure audit

Open one granule from S3 and inspect its HDF5 structure. Do this before running the
batch extractor. VNP14IMG v002 stores fire pixels differently from v001.

In [ ]:
# ── Open one granule and walk the HDF5 tree ───────────────────────────────────
# Pull from a confirmed high-fire day: 2020-09-05 (Creek Fire ignition)

AUDIT_DATE = '2020-09-05'

audit_granules = earthaccess.search_data(
    short_name   = VIIRS_SHORT_NAME,
    version      = VIIRS_VERSION,
    temporal     = (AUDIT_DATE, AUDIT_DATE),
    bounding_box = BBOX_TUPLE,
    count        = 2,
)

print(f"Found {len(audit_granules)} granules on {AUDIT_DATE}")

if audit_granules:
    print(f"Opening: {audit_granules[0]['meta']['native-id']}\n")

    def walk_h5(h5file, prefix='', depth=0, max_depth=4):
        if depth >= max_depth:
            return
        for key in h5file.keys():
            item = h5file[key]
            path = f"{prefix}/{key}"
            if isinstance(item, h5py.Dataset):
                print(f"  {'DATASET':<10} {path:<65} shape={item.shape}  dtype={item.dtype}")
            else:
                print(f"  {'GROUP':<10} {path}")
                walk_h5(item, path, depth + 1, max_depth)

    with earthaccess.open([audit_granules[0]]) as fhs:
        with h5py.File(fhs[0], 'r') as hf:
            print("═══ VNP14IMG v002 HDF5 structure ═══")
            walk_h5(hf)
            print("\n═══ Global attributes (first 12) ═══")
            for k in list(hf.attrs)[:12]:
                print(f"  {k}: {hf.attrs[k]}")

In [ ]:
# ── VNP14IMG v002 variable paths ──────────────────────────────────────────────
# UPDATE these paths after inspecting the structure audit above.
# The paths shown here are the expected v002 structure — verify against your audit output.
#
# VNP14IMG v002 stores fire pixels in a "fire pixel" group:
#   phony_dim_0 = number of detected fire pixels in granule (varies 0–N)
#   phony_dim_1 x phony_dim_2 = full swath grid (6464 x 6400 pixels)
#
# We extract from the FIRE PIXEL arrays (phony_dim_0), not the full swath.
# Full swath extraction is done in Module 1c.

# Fire pixel variables (one value per detected fire pixel)
FP_VARS = {
    'latitude':   'Fire Pixels/FP_latitude',
    'longitude':  'Fire Pixels/FP_longitude',
    'BT_I4':      'Fire Pixels/FP_T4',          # I4 brightness temperature (K)
    'BT_I5':      'Fire Pixels/FP_T5',          # I5 brightness temperature (K)
    'BT_I4_bg':   'Fire Pixels/FP_MeanT4',      # local background mean BT_I4
    'BT_I5_bg':   'Fire Pixels/FP_MeanT5',      # local background mean BT_I5
    'MAD_I4':     'Fire Pixels/FP_MAD_T4',      # local background MAD of BT_I4
    'MAD_I5':     'Fire Pixels/FP_MAD_T5',      # local background MAD of BT_I5
    'frp_mw':     'Fire Pixels/FP_power',       # fire radiative power (MW)
    'confidence': 'Fire Pixels/FP_confidence',  # 8=nominal, 9=high
    'sol_zen':    'Fire Pixels/FP_SolZenAng',   # solar zenith angle (degrees)
    'view_zen':   'Fire Pixels/FP_ViewZenAng',  # view zenith angle (degrees)
    'BT_diff_bg': 'Fire Pixels/FP_MeanDT',      # mean background delta-T
    'is_day':     'Fire Pixels/FP_day',         # 1=day, 0=night
}

# Fill values used in VNP14IMG v002 (mask these before computing derived features)
FP_FILL = {
    'BT_I4':    -999.0,
    'BT_I5':    -999.0,
    'frp_mw':   -999.0,
    'sol_zen':  -999.0,
    'view_zen': -999.0,
}

print("✓ FP_VARS defined")
print("  → Update paths above if structure audit shows different group/variable names")

---
## Section 6 — Single-granule fire pixel extractor

In [ ]:
def extract_granule_fp(
    file_handle,
    granule,
    bbox_dict: dict,
    fp_vars: dict,
    fp_fill: dict,
) -> pd.DataFrame:
    """
    Extract fire pixel records from a single VNP14IMG granule.

    Reads the fire pixel arrays (phony_dim_0 dimension), applies spatial
    filtering to the bounding box, masks fill values, and computes three
    derived features: BTD, BT_I4_anom, BTD_anom.

    Parameters
    ----------
    file_handle : S3 streaming file object from earthaccess.open()
    granule     : earthaccess DataGranule object (for metadata)
    bbox_dict   : dict with lon_min, lon_max, lat_min, lat_max
    fp_vars     : dict mapping output column names to HDF5 paths
    fp_fill     : dict mapping column names to fill values to mask as NaN

    Returns
    -------
    pd.DataFrame with one row per fire pixel, or empty DataFrame if none
    in bbox. Columns: all fp_vars keys + BTD, BT_I4_anom, BTD_anom,
    date, granule_id.
    """
    records = {}

    with warnings.catch_warnings():
        warnings.simplefilter('ignore')  # suppress xarray phony_dims UserWarning
        with h5py.File(file_handle, 'r') as hf:

            # ── Check fire pixel count ────────────────────────────────────────
            try:
                lat_arr = hf[fp_vars['latitude']][:].astype(np.float32)
            except KeyError:
                # Path mismatch — update FP_VARS after structure audit
                return pd.DataFrame()

            if lat_arr.size == 0:
                return pd.DataFrame()  # no fire pixels in this granule

            lon_arr = hf[fp_vars['longitude']][:].astype(np.float32)

            # ── Spatial filter ────────────────────────────────────────────────
            in_bbox = (
                (lon_arr >= bbox_dict['lon_min']) & (lon_arr <= bbox_dict['lon_max']) &
                (lat_arr >= bbox_dict['lat_min']) & (lat_arr <= bbox_dict['lat_max'])
            )

            if not in_bbox.any():
                return pd.DataFrame()

            records['latitude']  = lat_arr[in_bbox]
            records['longitude'] = lon_arr[in_bbox]

            # ── Read remaining variables ──────────────────────────────────────
            for col, h5path in fp_vars.items():
                if col in ('latitude', 'longitude'):
                    continue
                try:
                    arr = hf[h5path][:].astype(np.float32)[in_bbox]
                    # Mask fill values
                    if col in fp_fill:
                        arr = np.where(arr == fp_fill[col], np.nan, arr)
                    records[col] = arr
                except KeyError:
                    records[col] = np.full(in_bbox.sum(), np.nan)

    # ── Build DataFrame ───────────────────────────────────────────────────────
    df = pd.DataFrame(records)

    # ── Derived features ──────────────────────────────────────────────────────
    # BTD: the primary fire spectral signal. Fire pixels show BTD >> 0.
    # Non-fire emitters (urban, industrial) typically show BTD < 15 K.
    df['BTD'] = df['BT_I4'] - df['BT_I5']

    # BT_I4_anom: how much hotter is this pixel vs its local background?
    # Captures fire intensity relative to ambient surface temperature.
    df['BT_I4_anom'] = df['BT_I4'] - df['BT_I4_bg']

    # BTD_anom: fire spectral contrast above background spectral contrast.
    # Suppresses persistent warm surfaces (deserts, UHI) that inflate BTD.
    df['BTD_anom'] = df['BTD'] - df['BT_diff_bg']

    # ── Metadata columns ─────────────────────────────────────────────────────
    try:
        time_str = granule['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']
        date_str = time_str[:10]
    except (KeyError, TypeError):
        date_str = 'unknown'

    df['date']       = date_str
    df['granule_id'] = str(granule.get('meta', {}).get('native-id', 'unknown'))

    return df


print("✓ extract_granule_fp() defined")

---
## Section 7 — Quick test on a single day (2020-09-05)

Always run this before the full batch. Verify fire pixel counts and value ranges
look reasonable before committing to a 90–120 minute extraction run.

In [ ]:
# ── Test extraction on Creek Fire ignition day ─────────────────────────────
TEST_DATE = '2020-09-05'

test_granules = earthaccess.search_data(
    short_name   = VIIRS_SHORT_NAME,
    version      = VIIRS_VERSION,
    temporal     = (TEST_DATE, TEST_DATE),
    bounding_box = BBOX_TUPLE,
    count        = 10,
)

print(f"Found {len(test_granules)} granules on {TEST_DATE}")

test_frames = []
with earthaccess.open(test_granules[:5]) as fhs:
    for granule, fh in zip(test_granules[:5], fhs):
        df = extract_granule_fp(fh, granule, BBOX_DICT, FP_VARS, FP_FILL)
        if len(df) > 0:
            test_frames.append(df)

if test_frames:
    test_df = pd.concat(test_frames, ignore_index=True)
    print(f"\nTest result: {len(test_df):,} fire pixels from {len(test_frames)} granules")
    print(f"\nColumn summary:")
    print(test_df[['BT_I4', 'BT_I5', 'BTD', 'frp_mw', 'view_zen', 'confidence']].describe().round(2))

    # Verification checks from the team guide
    print("\n── Verification checks ──")
    bt_ok  = (test_df['BT_I4'].dropna().between(208, 367)).mean()
    btd_ok = (test_df['BTD'].dropna() > 0).mean()
    frp_ok = (test_df['frp_mw'].dropna().between(0.08, 2100)).mean()
    print(f"BT_I4 in 208–367 K:    {bt_ok*100:.1f}%  (expect ~100%)")
    print(f"BTD > 0:               {btd_ok*100:.1f}%  (expect >70%)")
    print(f"FRP in 0.08–2100 MW:   {frp_ok*100:.1f}%  (expect ~100%)")
else:
    print("No fire pixels found — check FP_VARS paths match structure audit")

In [ ]:
# ── Visualise test day fire pixels ────────────────────────────────────────────
if test_frames:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    sc0 = axes[0].scatter(test_df['longitude'], test_df['latitude'],
                          c=test_df['BT_I4'], cmap='inferno', s=4, alpha=0.7,
                          vmin=310, vmax=367)
    plt.colorbar(sc0, ax=axes[0], label='K')
    axes[0].set_title(f'BT_I4 (MWIR) — {TEST_DATE}')
    axes[0].set_xlabel('Longitude');  axes[0].set_ylabel('Latitude')

    sc1 = axes[1].scatter(test_df['longitude'], test_df['latitude'],
                          c=test_df['BTD'], cmap='hot', s=4, alpha=0.7,
                          vmin=0, vmax=80)
    plt.colorbar(sc1, ax=axes[1], label='K')
    axes[1].set_title('BTD = BT_I4 − BT_I5')
    axes[1].set_xlabel('Longitude')

    sc2 = axes[2].scatter(test_df['longitude'], test_df['latitude'],
                          c=np.log10(test_df['frp_mw'].clip(0.1)),
                          cmap='YlOrRd', s=4, alpha=0.7)
    plt.colorbar(sc2, ax=axes[2], label='log₁₀(MW)')
    axes[2].set_title('FRP (log scale)')
    axes[2].set_xlabel('Longitude')

    fig.suptitle(f'FireSight-IR | Module 1a — VNP14IMG fire pixels, {TEST_DATE}', fontsize=10)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '01a_test_day_fire_pixels.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✓ Figure saved to Drive")

---
## Section 8 — Full batch processor

Processes all VIIRS granules for active fire days across 2018–2023.

**Runtime:** ~90–120 minutes total. 2020 and 2021 are the largest years.  
**If Colab disconnects:** Reconnect, rerun Sections 0–3, then rerun this cell. Completed years are skipped automatically.

In [ ]:
def process_year(
    year: int,
    firms_dir: Path,
    output_dir: Path,
    bbox_dict: dict,
    bbox_tuple: tuple,
    short_name: str,
    version: str,
    fp_vars: dict,
    fp_fill: dict,
    min_firms_count: int = 10,
    granule_batch_size: int = 10,
) -> dict:
    """
    Process all active fire days for a single year.

    For each active fire day:
      1. Query CMR for VNP14IMG granules
      2. Stream granules from S3 in batches
      3. Extract fire pixel records
      4. Concatenate and write annual parquet to output_dir

    Returns a summary dict with year, n_pixels, n_days, n_errors.
    """
    out_path = output_dir / f'viirs_fp_{year}.parquet'

    if out_path.exists():
        df_existing = pd.read_parquet(out_path)
        print(f"  {year}: {len(df_existing):>9,} pixels  [already on Drive — skipped]")
        return {'year': year, 'n_pixels': len(df_existing), 'n_days': -1, 'n_errors': 0}

    active_days = get_active_fire_days(year, firms_dir, min_firms_count)
    if not active_days:
        print(f"  {year}: no active fire days found")
        return {'year': year, 'n_pixels': 0, 'n_days': 0, 'n_errors': 0}

    year_frames = []
    n_errors = 0

    for day_str in tqdm(active_days, desc=f'{year}', unit='day', leave=False):
        try:
            day_granules = earthaccess.search_data(
                short_name   = short_name,
                version      = version,
                temporal     = (day_str, day_str),
                bounding_box = bbox_tuple,
                count        = 50,  # ~6–8 granules per day over study region
            )

            if not day_granules:
                continue

            # Process in batches to manage S3 connection overhead
            for i in range(0, len(day_granules), granule_batch_size):
                batch = day_granules[i:i + granule_batch_size]
                with earthaccess.open(batch) as fhs:
                    for granule, fh in zip(batch, fhs):
                        try:
                            df = extract_granule_fp(fh, granule, bbox_dict, fp_vars, fp_fill)
                            if len(df) > 0:
                                year_frames.append(df)
                        except Exception:
                            n_errors += 1

        except Exception as day_err:
            n_errors += 1
            if n_errors <= 3:
                print(f"  [WARN] {day_str}: {day_err}")

    if not year_frames:
        print(f"  {year}: no fire pixels extracted")
        return {'year': year, 'n_pixels': 0, 'n_days': len(active_days), 'n_errors': n_errors}

    # Concatenate, deduplicate (same pixel may appear in overlapping granules)
    year_df = pd.concat(year_frames, ignore_index=True)
    year_df = year_df.drop_duplicates(subset=['latitude', 'longitude', 'date'])

    year_df.to_parquet(out_path, index=False)
    return {'year': year, 'n_pixels': len(year_df), 'n_days': len(active_days), 'n_errors': n_errors}


print("✓ process_year() defined")

In [ ]:
# ── Run full extraction — 2018 to 2023 ────────────────────────────────────────
# Expect ~90–120 minutes total. 0% progress bars during S3 connection retries
# are normal — do not interrupt the cell.

extraction_log = []

for year in ALL_YEARS:
    flag = ' ← VAL YEAR' if year == VAL_YEAR else ''
    print(f"\n── {year}{flag} ──")
    summary = process_year(
        year             = year,
        firms_dir        = FIRMS_DIR,
        output_dir       = VIIRS_DIR,
        bbox_dict        = BBOX_DICT,
        bbox_tuple       = BBOX_TUPLE,
        short_name       = VIIRS_SHORT_NAME,
        version          = VIIRS_VERSION,
        fp_vars          = FP_VARS,
        fp_fill          = FP_FILL,
        min_firms_count  = MIN_FIRMS_COUNT,
    )
    extraction_log.append(summary)
    if summary['n_days'] != -1:  # not skipped
        print(f"  {year}: {summary['n_pixels']:>9,} fire pixels | "
              f"{summary['n_days']} active days | "
              f"{summary['n_errors']} errors")

print("\n✓ Full extraction complete")

---
## Section 9 — Output verification

Run all checks from the team guide verification checklist.

In [ ]:
# ── Verification checklist ────────────────────────────────────────────────────
print("═══ FireSight-IR Module 1a — Output Verification ═══\n")

log_df = pd.DataFrame(extraction_log)

# Check 1: Expected year-by-year pixel counts (from team guide)
EXPECTED_PIXELS = {
    2018: (150_000, 250_000),   # ~197k expected
    2019: (40_000,  90_000),    # ~62k expected
    2020: (250_000, 450_000),   # ~358k expected — record year
    2021: (250_000, 450_000),   # ~358k expected — record year
    2022: (60_000,  150_000),   # ~98k expected
    2023: (40_000,  130_000),   # ~76k expected (val year)
}

print(f"{'Year':<8} {'Pixels':>12} {'Status':<10} {'Expected range':<25} {'Note'}")
print('─' * 80)

for yr in ALL_YEARS:
    path = VIIRS_DIR / f'viirs_fp_{yr}.parquet'
    if not path.exists():
        print(f"{yr:<8} {'MISSING':>12} {'FAIL':<10}")
        continue

    df = pd.read_parquet(path)
    n  = len(df)
    lo, hi = EXPECTED_PIXELS.get(yr, (0, 9e9))
    ok = 'PASS' if lo <= n <= hi else 'CHECK'
    note = '← VAL YEAR' if yr == VAL_YEAR else ''
    print(f"{yr:<8} {n:>12,} {ok:<10} {lo:,} – {hi:,}        {note}")

    # Value range checks on 2020 (largest year, most diagnostic)
    if yr == 2020:
        bt_ok  = df['BT_I4'].dropna().between(208, 367).mean()
        btd_ok = (df['BTD'].dropna() > 0).mean()
        frp_ok = df['frp_mw'].dropna().between(0.08, 2100).mean()
        print(f"  2020 range checks:")
        print(f"    BT_I4 in 208–367 K : {bt_ok*100:.1f}%  (expect ~100%)")
        print(f"    BTD > 0            : {btd_ok*100:.1f}%  (expect >70%)")
        print(f"    FRP in range       : {frp_ok*100:.1f}%  (expect ~100%)")

# Check 2: Total pixels
total_pixels = sum(
    len(pd.read_parquet(VIIRS_DIR / f'viirs_fp_{yr}.parquet'))
    for yr in TRAIN_YEARS
    if (VIIRS_DIR / f'viirs_fp_{yr}.parquet').exists()
)
print(f"\nTotal training pixels (2018–2022): {total_pixels:,}")
print(f"  Expect >1,000,000 — {'PASS' if total_pixels > 1_000_000 else 'CHECK'}")

# Check 3: 2020 and 2021 are the largest years
pixel_counts = {}
for yr in ALL_YEARS:
    p = VIIRS_DIR / f'viirs_fp_{yr}.parquet'
    if p.exists():
        pixel_counts[yr] = len(pd.read_parquet(p))

if pixel_counts:
    top2 = sorted(pixel_counts, key=pixel_counts.get, reverse=True)[:2]
    extreme_ok = set(top2) <= {2020, 2021}
    print(f"\nTop 2 years by pixel count: {top2}")
    print(f"  2020 and 2021 highest — {'PASS' if extreme_ok else 'CHECK (unexpected)'}")

# Check 4: Drive parquet file count
n_files = len(list(FIRMS_DIR.glob('*.parquet'))) + len(list(VIIRS_DIR.glob('*.parquet')))
print(f"\nParquet files on Drive: {n_files}  (expect 12 = 6 FIRMS + 6 VIIRS)")
print(f"  {'PASS' if n_files == 12 else f'CHECK — {n_files}/12 files present'}")

In [ ]:
# ── Annual pixel count chart ───────────────────────────────────────────────────
if pixel_counts:
    fig, ax = plt.subplots(figsize=(9, 4))
    years_x = list(pixel_counts.keys())
    counts  = list(pixel_counts.values())
    colors  = ['#A32D2D' if yr == VAL_YEAR else '#378ADD' for yr in years_x]

    bars = ax.bar(years_x, counts, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)

    for bar, n in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
                f'{n/1e3:.0f}k', ha='center', va='bottom', fontsize=9)

    ax.set_xlabel('Year')
    ax.set_ylabel('Fire pixels')
    ax.set_title('FireSight-IR | Module 1a — VNP14IMG fire pixels per year\n'
                 'Blue = training | Red = validation (held out)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e3:.0f}k'))

    # Annotate notable fire events
    annotations = {
        2018: 'Camp Fire\nMendocino Complex',
        2020: 'Creek Fire\nAugust Complex\n(record season)',
        2021: 'Dixie Fire\nCaldor Fire',
    }
    for yr, label in annotations.items():
        if yr in pixel_counts:
            ax.annotate(label, xy=(yr, pixel_counts[yr]),
                       xytext=(yr, pixel_counts[yr] + max(counts)*0.06),
                       ha='center', fontsize=7.5, color='#333333',
                       arrowprops=dict(arrowstyle='-', color='gray', lw=0.8))

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '01a_pixel_counts.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Figure saved to Drive")

---
## Section 10 — FIRMS type column audit (label construction preview)

The `type` column is critical for Module 2 label construction. Inspect the distribution
across years to confirm false-alarm training examples (type=2) are present.

In [ ]:
# ── Audit FIRMS type distribution across all years ────────────────────────────
print("FIRMS type column — label construction preview")
print("type=0: vegetation fire → positive wildfire label")
print("type=2: static land source (gas flares, industrial) → false-alarm training")
print("type=3: offshore → exclude from training")
print()

type_rows = []
for yr in ALL_YEARS:
    path = FIRMS_DIR / f'firms_viirs_snpp_{yr}.parquet'
    if not path.exists():
        continue
    df = pd.read_parquet(path, columns=['type'])
    counts = df['type'].value_counts().to_dict()
    total  = len(df)
    type_rows.append({
        'year':    yr,
        'type_0':  counts.get(0, 0),
        'type_2':  counts.get(2, 0),
        'type_3':  counts.get(3, 0),
        'total':   total,
        'pct_veg': 100 * counts.get(0, 0) / max(total, 1),
    })

type_df = pd.DataFrame(type_rows)
print(type_df.to_string(index=False))

# Confirm type=2 exists as false-alarm training pool
total_type2 = type_df['type_2'].sum()
print(f"\nTotal type=2 detections (false-alarm pool): {total_type2:,}")
print(f"  {'Good — sufficient false-alarm training examples' if total_type2 > 1000 else 'Low — may need to supplement with DNB/OSM masks in Module 2'}")

---
## Section 11 — Summary

### Output files on Google Drive

| File | Path | Description |
|---|---|---|
| `firms_viirs_snpp_{YEAR}.parquet` | `data/raw/firms/` | Ground truth labels — one row per fire detection |
| `viirs_fp_{YEAR}.parquet` | `data/processed/viirs_fp/` | Primary ML inputs — one row per fire pixel with BT, FRP, derived features |
| `01a_test_day_fire_pixels.png` | `figures/` | Spatial map of BT_I4, BTD, FRP for 2020-09-05 |
| `01a_pixel_counts.png` | `figures/` | Annual fire pixel counts |

### Column reference for viirs_fp parquet files

| Column | Source | Description |
|---|---|---|
| `BT_I4`, `BT_I5` | VNP14IMG | Brightness temperature in K — FireSat-analog MWIR/LWIR |
| `BT_I4_bg`, `BT_I5_bg` | VNP14IMG | Local background mean BT |
| `MAD_I4`, `MAD_I5` | VNP14IMG | Local background mean absolute deviation |
| `frp_mw` | VNP14IMG | Fire radiative power in MW |
| `confidence` | VNP14IMG | 8=nominal, 9=high |
| `sol_zen`, `view_zen` | VNP14IMG | Solar and view zenith angles — critical for Beer-Lambert |
| `BTD` | derived | BT_I4 − BT_I5 — primary fire spectral signal |
| `BT_I4_anom` | derived | BT_I4 − BT_I4_bg — fire intensity above ambient |
| `BTD_anom` | derived | BTD − BT_diff_bg — fire contrast above background |

### What comes next — Module 1b

Module 1b downloads ERA5 atmospheric profiles co-located with the fire pixels from this notebook.
For each fire detection location and date, it retrieves temperature profile, specific humidity,
and PBL height. These become inputs to the atmospheric MLP branch and the Beer-Lambert
transmittance calculations in the physics-informed loss function.

**Before Module 1b:** Register for a free Copernicus CDS account at  
https://cds.climate.copernicus.eu/ and install your CDS API key.